<a href="https://colab.research.google.com/github/evkoff/DI-Bootcamp-Stage1/blob/main/Week13/Day3/ExerciseXP/W13Dd_XP_Sentiment_Assistant_with_BERT_Fine_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sentiment Assistant with BERT Fine-Tuning (PyTorch)
## 0. Engineering Note

The original assignment uses the TensorFlow implementation of BERT (`TFBertForSequenceClassification`).

In the current Google Colab environment, the installed version of the Hugging Face `transformers` library (5.x) no longer provides this TensorFlow model. Therefore, the project is implemented using the equivalent PyTorch model (`BertForSequenceClassification`), while preserving the same preprocessing, fine-tuning, evaluation, and inference workflow.

Pipeline:

1. Imports
2. Hardware check
3. Load IMDB
4. Tokenizer
5. Dataset
6. DataLoader
7. Load pretrained BERT
8. Fine-tuning
9. Evaluation
10. Inference

## 1. Import Libraries and Check Hardware

In [1]:
# Standard library
import platform

# PyTorch
import torch
from torch.utils.data import DataLoader

# Hugging Face
from datasets import load_dataset #library Datasets is an official HuggingFace instrument for datasets loading
from transformers import (
    BertTokenizer,
    BertForSequenceClassification
)

print(f"Python version: {platform.python_version()}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Python version: 3.12.13
PyTorch version: 2.11.0+cu128
CUDA available: True
Device: cuda


## 2. Load and Explore the IMDB Dataset

In [2]:
import datasets
import huggingface_hub

print("datasets:", datasets.__version__)
print("huggingface_hub:", huggingface_hub.__version__)

datasets: 4.0.0
huggingface_hub: 1.23.0


In [3]:
!pip show datasets

Name: datasets
Version: 4.0.0
Summary: HuggingFace community-driven open-source library of datasets
Home-page: https://github.com/huggingface/datasets
Author: HuggingFace Inc.
Author-email: thomas@huggingface.co
License: Apache 2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: dill, filelock, fsspec, huggingface-hub, multiprocess, numpy, packaging, pandas, pyarrow, pyyaml, requests, tqdm, xxhash
Required-by: torchtune


In [4]:
from datasets import load_dataset

# Load the IMDB dataset
imdb = load_dataset("stanfordnlp/imdb")

print(imdb)

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [5]:
# Display dataset structure (splits)
print(imdb.keys())

# Display one training example
print(imdb["train"][0])

dict_keys(['train', 'test', 'unsupervised'])
{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really,

In [6]:
# make sure that there are no labels in the 'unsupervised'dataset (so we don't need this dataset further)
print(imdb["unsupervised"][0])
print(imdb["unsupervised"].unique("label"))

{'text': 'This is just a precious little diamond. The play, the script are excellent. I cant compare this movie with anything else, maybe except the movie "Leon" wonderfully played by Jean Reno and Natalie Portman. But... What can I say about this one? This is the best movie Anne Parillaud has ever played in (See please "Frankie Starlight", she\'s speaking English there) to see what I mean. The story of young punk girl Nikita, taken into the depraved world of the secret government forces has been exceptionally over used by Americans. Never mind the "Point of no return" and especially the "La femme Nikita" TV series. They cannot compare the original believe me! Trash these videos. Buy this one, do not rent it, BUY it. BTW beware of the subtitles of the LA company which "translate" the US release. What a disgrace! If you cant understand French, get a dubbed version. But you\'ll regret later :)', 'label': -1}
[-1]


## 3. Initialize the BERT Tokenizer


In [7]:
# Load the pretrained BERT tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
print(tokenizer)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

BertTokenizer(name_or_path='bert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})


## 4. Prepare the Dataset
Dataset, tokenization, encoding

### 4.0 Tokenization Function

In [8]:
# Tokenization function
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )

In [9]:
# Apply the tokenizer to a single example
sample = tokenize_function(imdb["train"][0])

# Display the result
print(sample)

{'input_ids': [101, 1045, 12524, 1045, 2572, 8025, 1011, 3756, 2013, 2026, 2678, 3573, 2138, 1997, 2035, 1996, 6704, 2008, 5129, 2009, 2043, 2009, 2001, 2034, 2207, 1999, 3476, 1012, 1045, 2036, 2657, 2008, 2012, 2034, 2009, 2001, 8243, 2011, 1057, 1012, 1055, 1012, 8205, 2065, 2009, 2412, 2699, 2000, 4607, 2023, 2406, 1010, 3568, 2108, 1037, 5470, 1997, 3152, 2641, 1000, 6801, 1000, 1045, 2428, 2018, 2000, 2156, 2023, 2005, 2870, 1012, 1026, 7987, 1013, 1028, 1026, 7987, 1013, 1028, 1996, 5436, 2003, 8857, 2105, 1037, 2402, 4467, 3689, 3076, 2315, 14229, 2040, 4122, 2000, 4553, 2673, 2016, 2064, 2055, 2166, 1012, 1999, 3327, 2016, 4122, 2000, 3579, 2014, 3086, 2015, 2000, 2437, 2070, 4066, 1997, 4516, 2006, 2054, 1996, 2779, 25430, 14728, 2245, 2055, 3056, 2576, 3314, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [10]:
print(len(sample["input_ids"]))
print(len(sample["attention_mask"]))

128
128


### 4.1 Apply Tokenization to the Entire Dataset

In [23]:
# Apply tokenization to the entire dataset
tokenized_imdb = imdb.map(
    tokenize_function,
    batched=True
)

In [24]:
print(tokenized_imdb)

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 50000
    })
})


## 5. Create a Custom PyTorch Dataset

In [25]:
# Custom PyTorch dataset
class IMDbDataset(torch.utils.data.Dataset):

    def __init__(self, dataset):
        # Store the Hugging Face dataset
        self.dataset = dataset

    def __len__(self):
        # Return the dataset size
        return len(self.dataset)

    def __getitem__(self, idx):
        # Get one example
        item = self.dataset[idx]

        # Convert lists to PyTorch tensors
        return {
            "input_ids": torch.tensor(item["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(item["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(item["label"], dtype=torch.long),
        }

In [26]:
# Create PyTorch datasets
train_dataset = IMDbDataset(tokenized_imdb["train"])
test_dataset = IMDbDataset(tokenized_imdb["test"])

In [27]:
print(tokenized_imdb["train"].format)

{'type': None, 'format_kwargs': {}, 'columns': ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'], 'output_all_columns': False}


In [28]:
# Get the first training example
sample = train_dataset[0]

# Display the keys
print(sample.keys())

# Display tensor shapes and data types
for key, value in sample.items():
    print(f"{key}: shape={value.shape}, dtype={value.dtype}")

dict_keys(['input_ids', 'attention_mask', 'labels'])
input_ids: shape=torch.Size([128]), dtype=torch.int64
attention_mask: shape=torch.Size([128]), dtype=torch.int64
labels: shape=torch.Size([]), dtype=torch.int64


##6. Create Dataset Loaders

In [29]:
# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False
)

In [30]:
# Get the first batch
batch = next(iter(train_loader))

# Display the batch keys
print(batch.keys())

# Display tensor shapes
for key, value in batch.items():
    print(f"{key}: {value.shape}")

dict_keys(['input_ids', 'attention_mask', 'labels'])
input_ids: torch.Size([8, 128])
attention_mask: torch.Size([8, 128])
labels: torch.Size([8])


## 7. Load the Pretrained BERT Model

In [31]:
# Load the pretrained BERT model
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

# Move the model to the selected device
model.to(device)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [32]:
print(model)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

## 8. Fine-Tune the Model

In [33]:
from transformers import get_scheduler

# Create the optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=5e-5
)

In [34]:
# Number of training epochs
num_epochs = 3

# Total number of training steps
num_training_steps = num_epochs * len(train_loader)

# Create the learning rate scheduler
lr_scheduler = get_scheduler(
    name="linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

In [35]:
# Train the model
model.train()

for epoch in range(num_epochs):

    print(f"Epoch {epoch + 1}/{num_epochs}")

    for batch in train_loader:

        # Move batch to the selected device
        batch = {k: v.to(device) for k, v in batch.items()}

        # Forward pass
        outputs = model(**batch)

        # Compute loss
        loss = outputs.loss

        # Clear gradients so gradients are not accumulated with each new batch
        optimizer.zero_grad()

        # Backpropagation
        loss.backward()

        # Update model parameters
        optimizer.step()


        # Update learning rate
        lr_scheduler.step()



    print(f"Loss: {loss.item():.4f}")

Epoch 1/3
Loss: 0.6901
Epoch 2/3
Loss: 0.4375
Epoch 3/3
Loss: 0.0022


## 9. Evaluate the Model

In [37]:
# Switch the model to evaluation mode
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for batch in test_loader:

        # Move batch to the selected device
        batch = {k: v.to(device) for k, v in batch.items()}

        # Forward pass
        outputs = model(**batch)

        # Get predicted class
        predictions = torch.argmax(outputs.logits, dim=1)

        # Count correct predictions
        correct += (predictions == batch["labels"]).sum().item()

        # Count total predictions
        total += batch["labels"].size(0)

accuracy = correct / total

print(f"Test Accuracy: {accuracy:.4f}")

Test Accuracy: 0.8812


## 10. Run Inference

In [40]:
def predict(text):
    # Switch the model to evaluation mode
    model.eval()

    # Tokenize the input text
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128,
    )

    # Move tensors to the selected device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Disable gradient computation
    with torch.no_grad():
        outputs = model(**inputs)

    # Get predicted class
    prediction = torch.argmax(outputs.logits, dim=1).item()

    # Convert class index to label
    label = "Positive" if prediction == 1 else "Negative"

    print(f"Review: {text}")
    print(f"Prediction: {label}")

In [41]:
predict("I really enjoyed this movie!")
predict("This film was terrible.")

Review: I really enjoyed this movie!
Prediction: Positive
Review: This film was terrible.
Prediction: Negative


## 11. Conclusions

In this project, a pre-trained BERT model (bert-base-uncased) was fine-tuned for binary sentiment classification using the IMDB movie reviews dataset.

The workflow included loading and preprocessing the dataset, tokenizing the text with the BERT tokenizer, creating custom PyTorch datasets and data loaders, loading a pre-trained transformer model, configuring the optimizer and learning rate scheduler, and training the model using the PyTorch training loop.

After training, the model was evaluated on the test dataset to measure its classification performance. Finally, inference was performed on new text examples to demonstrate the model's ability to predict the sentiment of previously unseen reviews.

This project demonstrates the complete fine-tuning pipeline for transformer-based text classification using Hugging Face Transformers and PyTorch. It also illustrates the advantage of transfer learning, where a pre-trained language model can be adapted to a new task with relatively little additional training while achieving strong performance.

The model achieved a test accuracy of 88.1%, demonstrating the effectiveness of transfer learning with pre-trained transformer models for sentiment analysis.
